# H5: Metacognitive Monitoring (Bayesian)

- H5a: Calibration → optimality (LOO comparison)
- H5b: Anxiety slope → choice shift
- H5c: ω → confidence, not anxiety
- H5d: Confidence → error type

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import bambi as bmb, arviz as az
import statsmodels.api as sm
from scipy.stats import pearsonr, zscore, linregress
from pathlib import Path

%matplotlib inline
REPO = Path(os.getcwd())
for _ in range(5):
    if (REPO / '.git').exists(): break
    REPO = REPO.parent
os.chdir(REPO)
EXCLUDE = [154,197,208]
DATA_DIR = Path('data/exploratory_350/processed/stage5_filtered_data_20260320_191950')
BKW = dict(draws=2000, tune=1000, chains=4, progressbar=False, random_seed=42)

## Load params + build master

In [2]:
# Load CM2 + MCMC params
master = pd.read_csv('results/stats/joint_optimal/master_subject_df.csv', index_col='subj')
mp = pd.read_csv('results/stats/joint_optimal/mcmc_m5_params.csv').set_index('subj')
shared = sorted(set(master.index) & set(mp.index))
master.loc[shared,'omega'] = mp.loc[shared,'omega']
master.loc[shared,'kappa'] = mp.loc[shared,'kappa']
master['log_omega'] = np.log(master['omega']); master['log_kappa'] = np.log(master['kappa'])
master['omega_z'] = zscore(master['log_omega'].values)
master['kappa_z'] = zscore(master['log_kappa'].values)

# Affect
beh = pd.read_csv(DATA_DIR/'behavior_rich.csv', low_memory=False)
beh = beh[~beh['subj'].isin(EXCLUDE)]
cdf = beh[beh['type']==1].copy(); cdf['T_round'] = cdf['threat'].round(1)
feelings = pd.read_csv(DATA_DIR/'feelings.csv'); feelings = feelings[~feelings['subj'].isin(EXCLUDE)]
cells = pd.read_csv('results/stats/vigor_analysis/cell_means.csv')
cells = cells[~cells['subj'].isin(EXCLUDE)]
anx = feelings[feelings['questionLabel']=='anxiety'][['subj','threat','response']].dropna()
conf = feelings[feelings['questionLabel']=='confidence'][['subj','threat','response']].dropna()
subjects = master.index.tolist()
for s in subjects:
    sa=anx[anx['subj']==s]; sc=conf[conf['subj']==s]
    if len(sa)>=5:
        sl,_,r_cal,_,_=linregress(sa['threat'],sa['response'])
        master.loc[s,'mean_anx']=sa['response'].mean()
        master.loc[s,'anx_slope']=sl; master.loc[s,'calibration']=r_cal
    if len(sc)>=5: master.loc[s,'mean_conf']=sc['response'].mean()

# Behavioral indices
attack = beh[beh['isAttackTrial']==1]
master['escape_rate'] = attack.groupby('subj').apply(lambda x:(x['trialEndState']=='escaped').mean())
master['earnings'] = beh.groupby('subj')['trialReward'].sum()
p_T01 = cdf[cdf['T_round']==0.1].groupby('subj')['choice'].mean()
p_T09 = cdf[cdf['T_round']==0.9].groupby('subj')['choice'].mean()
master['choice_shift'] = p_T01 - p_T09

# Optimality
opt_map = {}
for (T,D),grp in cdf.groupby(['T_round','distance_H']):
    er_h=grp[grp['choice']==1]['trialReward'].mean() if (grp['choice']==1).sum()>0 else -99
    er_l=grp[grp['choice']==0]['trialReward'].mean() if (grp['choice']==0).sum()>0 else -99
    opt_map[(T,D)]=1 if er_h>er_l else 0
cdf['optimal']=cdf.apply(lambda r:opt_map.get((r['T_round'],r['distance_H']),np.nan),axis=1)
cdf['is_opt']=(cdf['choice']==cdf['optimal']).astype(int)
cdf['err_type']=np.where(cdf['is_opt']==1,'correct',np.where(cdf['choice']==0,'overcautious','reckless'))
so=cdf.groupby('subj').agg(pct_opt=('is_opt','mean'),n_oc=('err_type',lambda x:(x=='overcautious').sum()),
    n_rk=('err_type',lambda x:(x=='reckless').sum()))
for c in so.columns:
    if c in master.columns: master=master.drop(columns=[c])
master=master.join(so,how='left')
print(f'Master: {len(master)} subjects (MCMC params)')

Master: 290 subjects (MCMC params)


## H5a: Calibration → optimality (LOO comparison)

In [3]:
v = master.dropna(subset=['omega_z','kappa_z','calibration','pct_opt','escape_rate','earnings'])
v['cal_z'] = zscore(v['calibration'].values)
print(f'H5a — Calibration beyond omega+kappa (N={len(v)})\n')
results_h5a = {}
for outcome, label in [('pct_opt','Optimality'),('escape_rate','Escape'),('earnings','Earnings')]:
    df = v[['omega_z','kappa_z','cal_z',outcome]].dropna()
    m_base = bmb.Model(f'{outcome} ~ omega_z + kappa_z', data=df)
    r_base = m_base.fit(**BKW, idata_kwargs={"log_likelihood": True})
    m_full = bmb.Model(f'{outcome} ~ omega_z + kappa_z + cal_z', data=df)
    r_full = m_full.fit(**BKW, idata_kwargs={"log_likelihood": True})
    loo_b = az.loo(r_base); loo_f = az.loo(r_full)
    delta = loo_f.elpd_loo - loo_b.elpd_loo
    s_f = az.summary(r_full, hdi_prob=0.95)
    lo = s_f.loc['cal_z','hdi_2.5%']; hi = s_f.loc['cal_z','hdi_97.5%']
    passed = lo > 0 or hi < 0
    results_h5a[outcome] = passed
    print(f'  {label}: ΔELPD={delta:+.1f}, cal β={s_f.loc["cal_z","mean"]:+.4f} [{lo:+.4f},{hi:+.4f}] {"✓" if passed else "✗"}')
n_pass = sum(results_h5a.values())
print(f'\nH5a: {"PASS ✓" if n_pass >= 2 else "FAIL ✗"} ({n_pass}/3)')

Initializing NUTS using jitter+adapt_diag...


H5a — Calibration beyond omega+kappa (N=286)



Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, omega_z, kappa_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


...

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, omega_z, kappa_z, cal_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


  Earnings: ΔELPD=+7.6, cal β=+21.0330 [+11.3670,+31.2430] ✓

H5a: PASS ✓ (3/3)


## H5b: Anxiety slope → choice shift

In [4]:
v = master.dropna(subset=['anx_slope','choice_shift'])
v['slope_z'] = zscore(v['anx_slope'].values)
m = bmb.Model('choice_shift ~ slope_z', data=v)
r = m.fit(**BKW)
s = az.summary(r, hdi_prob=0.95)
lo_b = s.loc['slope_z','hdi_2.5%']
p_b = lo_b > 0
print(f'H5b — slope → shift: β={s.loc["slope_z","mean"]:+.4f}, 95% HDI=[{lo_b:+.4f},{s.loc["slope_z","hdi_97.5%"]:+.4f}]')
print(f'H5b: {"PASS ✓" if p_b else "FAIL ✗"}')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, slope_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


H5b — slope → shift: β=+0.1160, 95% HDI=[+0.0840,+0.1470]
H5b: PASS ✓


## H5c: ω → confidence, not anxiety

In [5]:
v = master.dropna(subset=['omega_z','mean_anx','mean_conf'])
m_c = bmb.Model('mean_conf ~ omega_z', data=v)
r_c = m_c.fit(**BKW)
s_c = az.summary(r_c, hdi_prob=0.95)
hi_conf = s_c.loc['omega_z','hdi_97.5%']
pass_conf = hi_conf < 0

m_a = bmb.Model('mean_anx ~ omega_z', data=v)
r_a = m_a.fit(**BKW)
s_a = az.summary(r_a, hdi_prob=0.95)
lo_anx = s_a.loc['omega_z','hdi_2.5%']; hi_anx = s_a.loc['omega_z','hdi_97.5%']
pass_anx = lo_anx < 0 and hi_anx > 0  # includes zero

print(f'H5c — ω → conf: β={s_c.loc["omega_z","mean"]:+.3f} [{s_c.loc["omega_z","hdi_2.5%"]:+.3f},{hi_conf:+.3f}] {"✓" if pass_conf else "✗"}')
print(f'     ω → anx:  β={s_a.loc["omega_z","mean"]:+.3f} [{lo_anx:+.3f},{hi_anx:+.3f}] {"✓" if pass_anx else "✗"} (should include zero)')
print(f'H5c: {"PASS ✓" if pass_conf and pass_anx else "FAIL ✗"}')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, omega_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, omega_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


H5c — ω → conf: β=-0.288 [-0.440,-0.134] ✓
     ω → anx:  β=+0.089 [-0.059,+0.239] ✓ (should include zero)
H5c: PASS ✓


## H5d: Confidence → error type

In [6]:
v = master.dropna(subset=['mean_conf','n_oc','n_rk'])
v['conf_z'] = zscore(v['mean_conf'].values)
m_oc = bmb.Model('n_oc ~ conf_z', data=v)
r_oc = m_oc.fit(**BKW)
s_oc = az.summary(r_oc, hdi_prob=0.95)
p_oc = s_oc.loc['conf_z','hdi_97.5%'] < 0

m_rk = bmb.Model('n_rk ~ conf_z', data=v)
r_rk = m_rk.fit(**BKW)
s_rk = az.summary(r_rk, hdi_prob=0.95)
p_rk = s_rk.loc['conf_z','hdi_2.5%'] > 0

print(f'H5d — conf → OC: β={s_oc.loc["conf_z","mean"]:+.2f} [{s_oc.loc["conf_z","hdi_2.5%"]:+.2f},{s_oc.loc["conf_z","hdi_97.5%"]:+.2f}] {"✓" if p_oc else "✗"}')
print(f'     conf → rk: β={s_rk.loc["conf_z","mean"]:+.2f} [{s_rk.loc["conf_z","hdi_2.5%"]:+.2f},{s_rk.loc["conf_z","hdi_97.5%"]:+.2f}] {"✓" if p_rk else "✗"}')
print(f'H5d: {"PASS ✓" if p_oc and p_rk else "FAIL ✗"}')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, conf_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, conf_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


H5d — conf → OC: β=-1.60 [-2.45,-0.82] ✓
     conf → rk: β=+0.59 [+0.25,+0.91] ✓
H5d: PASS ✓


## Summary

In [7]:
print('H5 SUMMARY (Bayesian)')
print('='*60)
tests = [('H5a','Cal → optimality',results_h5a.get('pct_opt',False)),
         ('H5a','Cal → earnings',results_h5a.get('earnings',False)),
         ('H5b','Slope → shift',p_b),('H5c','ω → confidence',pass_conf),
         ('H5c','ω → anxiety null',pass_anx),('H5d','Conf → OC',p_oc),('H5d','Conf → reckless',p_rk)]
for h,t,p in tests: print(f'  {h:<6} {t:<22} {"✓" if p else "✗"}')
n=sum(1 for *_,p in tests if p)
print(f'\n{n}/{len(tests)} tests pass.')

H5 SUMMARY (Bayesian)
  H5a    Cal → optimality       ✓
  H5a    Cal → earnings         ✓
  H5b    Slope → shift          ✓
  H5c    ω → confidence         ✓
  H5c    ω → anxiety null       ✓
  H5d    Conf → OC              ✓
  H5d    Conf → reckless        ✓

7/7 tests pass.
